# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [37]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso 
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor


## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [38]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Laura.pkl')
#Prepare the train and test sets
X = df.drop(columns=['anneedeplantation'])
y = df['anneedeplantation']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=20)

categorical_variables = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

numerical_variables = X_train.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)

#pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])
	
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

['sous_categorie', 'adr_secteur', 'genre_bota', 'espece', 'portarbre', 'structure'] ['stadededeveloppement', 'hauteurarbre', 'latitude', 'longitude']


## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [39]:
# Initial cross-validation without hyperparameter tuning  
# We can also print the MSE or MAE scores from each cross-validation fold.

model_linear = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', LinearRegression())
])
model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(alpha=0.001, max_iter=10000))
])


model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(max_depth=10, min_samples_leaf=0.05,random_state=42))
])


In [40]:
arbres_scores = cross_val_score(model_linear, X_train, y_train, cv = 5, scoring='neg_mean_squared_error') 

print("MSE moyen :", np.mean(-arbres_scores))
print("Écart-type :", np.std(-arbres_scores))

MSE moyen : 139.5175592247019
Écart-type : 6.178005661137732


## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [41]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.

param_grid = [
 {'regression__alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

#arbres_scores_lasso = cross_val_score(model_lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
arbres_grid_search = GridSearchCV(model_lasso, param_grid, cv=5, scoring='neg_mean_squared_error')
arbres_grid_search.fit(X_train, y_train)

arbres_grid_results = pd.DataFrame(arbres_grid_search.cv_results_)

print(arbres_grid_results)
print(arbres_grid_search.best_estimator_)



   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0      13.077787      1.559408         0.025188        0.010446   
1       5.836213      0.601150         0.022996        0.005243   
2       2.109439      0.517948         0.019531        0.003653   
3       0.162483      0.022980         0.045738        0.034477   
4       0.116760      0.025880         0.018531        0.001923   
5       0.082546      0.012020         0.023179        0.003970   
6       0.066898      0.000545         0.018426        0.000874   
7       0.076387      0.007129         0.021530        0.001679   

   param_regression__alpha                        params  split0_test_score  \
0                    0.001  {'regression__alpha': 0.001}        -138.451303   
1                    0.010   {'regression__alpha': 0.01}        -144.572596   
2                    0.100    {'regression__alpha': 0.1}        -173.353017   
3                    1.000      {'regression__alpha': 1}        -221.274252   
4

In [ ]:
#GridSearch avec la random forest
param_grid_forest = {
    'regression__max_depth': [3, 4, 5, 10],
    'regression__min_samples_leaf': [0.1, 0.05],
    'regression__n_estimators': [50, 100, 200]
}

arbres_RF_grid_search = GridSearchCV(model_random_forest, param_grid_forest, cv=5, scoring='neg_mean_squared_error')
arbres_RF_grid_search.fit(X_train, y_train)

arbres_RF_grid_results = pd.DataFrame(arbres_RF_grid_search.cv_results_)

In [ ]:
print(arbres_RF_grid_results)
print(arbres_RF_grid_search.best_estimator_)

    mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0        0.635674      0.017859         0.027028        0.000929   
1        1.211544      0.015877         0.038133        0.001052   
2        2.490900      0.213171         0.068689        0.017184   
3        0.807728      0.010354         0.028382        0.001030   
4        1.545834      0.017397         0.038245        0.001892   
5        2.983948      0.038707         0.059721        0.001375   
6        0.732910      0.089366         0.028450        0.002457   
7        1.418326      0.014839         0.039861        0.001552   
8        2.818808      0.071957         0.061753        0.001457   
9        0.999983      0.015986         0.029128        0.001274   
10       1.893767      0.028228         0.041475        0.001171   
11       3.694785      0.038911         0.066752        0.002169   
12       0.732252      0.007739         0.028659        0.001580   
13       1.389533      0.013572         0.041716

## 5. Final Evaluation on the Test Set

In [ ]:
# Code for evaluating the best model on the test set.

# Créer un modèle de référence
dummy = DummyRegressor(strategy='mean')

# L'entraîner sur les données d'entraînement
dummy.fit(X_train_transformed, y_train)

# Prédire les années sur le test
y_pred_dummy = dummy.predict(X_test)

dummy_scores = cross_val_score(
    dummy,
    X_train_transformed,
    y_train,
    cv=10,
    scoring='neg_mean_squared_error'
)



In [ ]:
from sklearn.model_selection import cross_validate
import numpy as np

scoring = {
    'mse': 'neg_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# modèles Pipeline
models = {
    'LinearRegression': model_linear,
    'Lasso': model_lasso,
    'Dummy': dummy,
    'Random Forest' : model_random_forest
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    mse = -scores['test_mse']
    mae = -scores['test_mae']
    r2 = scores['test_r2']

    rmse = np.sqrt(mse)

    print(f"\n{name}")
    print(f"MSE moyen  : {mse.mean():.2f}")
    print(f"RMSE moyen : {rmse.mean():.2f}")
    print(f"MAE moyen  : {mae.mean():.2f}")
    print(f"R² moyen   : {r2.mean():.3f}")
    print(f"Écart-type MAE : {mae.std():.2f}")


LinearRegression
MSE moyen  : 139.52
RMSE moyen : 11.81
MAE moyen  : 8.92
R² moyen   : 0.575
Écart-type MAE : 0.13

Lasso
MSE moyen  : 138.95
RMSE moyen : 11.78
MAE moyen  : 8.95
R² moyen   : 0.577
Écart-type MAE : 0.13

Dummy
MSE moyen  : 328.57
RMSE moyen : 18.13
MAE moyen  : 15.90
R² moyen   : -0.000
Écart-type MAE : 0.11


KeyboardInterrupt: 

In [ ]:

#moyenne des mse pour le meilleur alpha --> répond à la question de k-fold, pour le meilleur alpha
#mse_cv = -arbres_grid_search.best_score_
#rmse_cv = np.sqrt(mse_cv)

#y_pred_test = arbres_grid_search.predict(X_test)
#mse_test = mean_squared_error(y_test, y_pred_test)
#rmse_test = np.sqrt(mse_test)

#mse_dummy = mean_squared_error(y_test, y_pred_dummy)

#mse_cv_dummy = -dummy_scores.mean()
#rmse_cv_dummy = np.sqrt(mse_cv_dummy)

# Comparaison
#print("Meilleur alpha :", arbres_grid_search.best_params_)
#estimation erreur de la validation croisée (k-fold)
#print("MSE validation croisée meilleur alpha :", mse_cv)
#print ('RMSE validation croisée meilleur alpha :',  rmse_cv)
#estimation erreur réelle sur le jeu de test
#print("MSE test :", mse_test)
#print ('RMSE test:',  rmse_test)
#estimation modèle simplifié
#print("MSE CV Dummy :", mse_cv_dummy)
#print("RMSE CV Dummy :", rmse_cv_dummy)
# Comparer les résultats
#print("MSE test Dummy :", mse_dummy)
#Pour info, différence entre les deux
#print("Différence des MSE avec meilleur alpha :", mse_test - mse_cv)
#print("Différence des RMSE avce meilleur alpha :", rmse_test - rmse_cv)

Meilleur alpha : {'regression__alpha': 0.001}
MSE validation croisée meilleur alpha : 138.95114893950387
RMSE validation croisée meilleur alpha : 11.787754194056808
MSE test : 145.5454854485922
RMSE test: 12.064223366988536
MSE CV Dummy : 328.55124943087486
RMSE CV Dummy : 18.12598271627982
MSE test Dummy : 335.6602019603576
Différence des MSE avec meilleur alpha : 6.594336509088322
Différence des RMSE avce meilleur alpha : 0.2764691729317281


## 6. Visualization of Predictions  
One model will suffice.

In [ ]:
# Visualisation of predictions.

## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [ ]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [ ]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [ ]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [ ]:
# Code

## Conclusion